In [3]:
import sqlite3
import pandas as pd
import numpy as np
import os
conn = sqlite3.connect('../data/sql/control_presupuestario.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

,name
0,departamentos
1,categorias
2,periodos
3,presupuesto
4,ejecucion_real_raw
5,ingresos


In [4]:
DB_PATH = "../data/sql/control_presupuestario.db"
PROCESSED_DIR = "../data/processed"

In [7]:
def cargar_datos_base(conn):
    return {
        "departamentos": pd.read_sql("SELECT * FROM departamentos", conn),
        
    }


In [10]:
df_periodos = pd.read_sql(
    "SELECT * FROM periodos",
    conn
)

if (df_periodos["anio"] == 2026).any():
    # El año 2026 ya existe; no insertamos duplicados
    print("Los períodos de 2026 ya existen.")

else:
    max_id = (
        int(df_periodos["periodo_id"].max())
        if not df_periodos.empty
        else 0
    )

    nuevos = [
        (max_id + i, 2026, i)
        for i in range(1, 13)
    ]

    conn.executemany(
        """
        INSERT INTO periodos (periodo_id, anio, mes)
        VALUES (?, ?, ?)
        """,
        nuevos
    )

    conn.commit()
    print("Se han insertado los 12 períodos de 2026.")

# Recargar el DataFrame con los datos actualizados
df_periodos = pd.read_sql(
    "SELECT * FROM periodos",
    conn
)

Se han insertado los 12 períodos de 2026.


In [15]:
df_periodos = pd.read_sql("SELECT * FROM periodos", conn)
p2025 = df_periodos[df_periodos["anio"] == 2025]

por_linea = (df_ingresos.merge(p2025, on="periodo_id")
             .groupby(["departamento_id", "concepto", "mes"])["monto_ingresos"]
             .sum().reset_index())

In [20]:
def forecast_ingresos(df_ingresos, df_periodos, mapa_periodo_2026):
    """Tendencia lineal + indice estacional + intervalo de confianza 95%."""
    p2025 = df_periodos[df_periodos["anio"] == 2025]
    por_linea = (df_ingresos.merge(p2025, on="periodo_id")
                 .groupby(["departamento_id", "concepto", "mes"])["monto_ingresos"]
                 .sum().reset_index())

    filas = []
    for (dept_id, concepto), grupo in por_linea.groupby(["departamento_id", "concepto"]):
        grupo = grupo.sort_values("mes")
        x, y = grupo["mes"].values, grupo["monto_ingresos"].values

        pendiente, intercepto = np.polyfit(x, y, 1)
        tendencia = pendiente * x + intercepto
        indice_estacional = y / tendencia
        error_estandar = np.std(y - tendencia, ddof=2)
        margen = NIVEL_CONFIANZA_Z * error_estandar

        x_2026 = np.arange(13, 25)
        forecast_vals = (pendiente * x_2026 + intercepto) * indice_estacional

        for mes, monto in zip(range(1, 13), forecast_vals):
            filas.append([dept_id, concepto, mapa_periodo_2026[mes],
                          round(monto, 2), round(monto - margen, 2), round(monto + margen, 2)])

    df = pd.DataFrame(filas, columns=["departamento_id", "concepto", "periodo_id",
                                       "monto_ingresos_forecast", "monto_min", "monto_max"])
    df.insert(0, "id_forecast", range(1, len(df) + 1))
    return df



In [21]:
def gasto_esperado_2026(df_presupuesto_2026, df_kpi, df_periodos):
    """Aplica el ratio de ejecucion historico POR MES (no el promedio anual)."""
    p2025 = df_periodos[df_periodos["anio"] == 2025]
    ratio_mensual = (df_kpi.merge(p2025, on="periodo_id")
                     [["departamento_id", "categoria_id", "mes", "pct_ejecucion"]])

    p2026 = df_periodos[df_periodos["anio"] == 2026]
    base = df_presupuesto_2026.merge(p2026, on="periodo_id")
    base = base.merge(ratio_mensual, on=["departamento_id", "categoria_id", "mes"], how="left")
    base["pct_ejecucion"] = base["pct_ejecucion"].fillna(1.0)
    base["monto_gasto_esperado"] = round(base["monto_presupuestado_2026"] * base["pct_ejecucion"], 2)

    df = base[["departamento_id", "categoria_id", "periodo_id", "monto_gasto_esperado"]].copy()
    df.insert(0, "id_gasto", range(1, len(df) + 1))
    return df

In [1]:
 print("=" * 60)
    print("FORECAST 2026 - RESUMEN")
    print("=" * 60)
    print(f"Ingresos      2025: {total_ing_2025:>14,.0f} EUR")
    print(f"Ingresos      2026: {total_ing_2026:>14,.0f} EUR  ({(total_ing_2026/total_ing_2025-1)*100:+.1f}%)")
    print(f"Presupuesto   2025: {total_pres_2025:>14,.0f} EUR")
    print(f"Presupuesto   2026: {total_pres_2026:>14,.0f} EUR  ({(total_pres_2026/total_pres_2025-1)*100:+.1f}%)")
    print(f"Gasto esperado 2026: {total_gasto_2026:>13,.0f} EUR")
    print(f"Margen operativo esperado 2026: {(1 - total_gasto_2026/total_ing_2026)*100:.1f}%")
    print("=" * 60)
    print("Tablas guardadas en SQLite y en data/processed/:")
    print("  - ingresos_forecast_2026.csv (incluye monto_min / monto_max, IC 95%)")
    print("  - presupuesto_2026.csv (variacion mensual preservada)")
    print("  - gasto_esperado_2026.csv (ratio de ejecucion mensual aplicado)")

    conn.close()


IndentationError: unexpected indent (3366934995.py, line 2)